# Preference-Guided Diffusion Steering - Exploration Notebook

This notebook provides exploration and analysis tools for the preference-guided diffusion steering project.

## Setup

In [ ]:
import sys
import os
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from preference_guided_diffusion_steering.models.model import PreferenceGuidedDiffusionModel
from preference_guided_diffusion_steering.data.loader import UltraFeedbackDataset
from preference_guided_diffusion_steering.utils.config import load_config

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete!")

## Configuration

In [ ]:
# Load configuration
config = load_config('../configs/default.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {device}")
print(f"Model: {config.get('model.base_model_path')}")

## Data Exploration

In [ ]:
# Load preference dataset
dataset = UltraFeedbackDataset(max_samples=100, seed=42)

print(f"Dataset size: {len(dataset)}")
if len(dataset) > 0:
    sample = dataset[0]
    print(f"Sample keys: {list(sample.keys())}")
    print(f"Sample prompt: {sample['prompt'][:100]}...")

In [ ]:
# Analyze preference data distribution
if len(dataset) > 0:
    rating_diffs = [pair.get('rating_diff', 0) for pair in dataset.preference_pairs]
    
    plt.figure(figsize=(10, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(rating_diffs, bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('Rating Difference')
    plt.ylabel('Frequency')
    plt.title('Distribution of Preference Rating Differences')
    
    plt.subplot(1, 2, 2)
    prompt_lengths = [len(pair['prompt'].split()) for pair in dataset.preference_pairs]
    plt.hist(prompt_lengths, bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('Prompt Length (words)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Prompt Lengths')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Average rating difference: {np.mean(rating_diffs):.2f}")
    print(f"Average prompt length: {np.mean(prompt_lengths):.1f} words")

## Model Analysis

In [ ]:
# Create model for analysis (mock version for exploration)
try:
    model = PreferenceGuidedDiffusionModel(
        base_model_path=config.get('model.base_model_path'),
        steering_config=config.get('model.steering_config'),
        device=device
    )
    print("Model loaded successfully!")
    
    # Analyze steering module parameters
    total_params = sum(p.numel() for p in model.steering_module.parameters())
    trainable_params = sum(p.numel() for p in model.steering_module.parameters() if p.requires_grad)
    
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
except Exception as e:
    print(f"Model loading failed (expected in exploration): {e}")
    print("Using mock analysis instead...")
    
    # Mock parameter analysis
    from preference_guided_diffusion_steering.models.model import SteeringModule
    
    steering = SteeringModule(**config.get('model.steering_config'))
    total_params = sum(p.numel() for p in steering.parameters())
    
    print(f"Steering module parameters: {total_params:,}")
    print(f"Guidance scale: {steering.get_guidance_scale():.4f}")

## Training Results Analysis

In [ ]:
# Load training history if available
history_path = Path('../results/training_history.json')

if history_path.exists():
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    print("Training history found!")
    
    # Plot training curves
    plt.figure(figsize=(15, 5))
    
    # Training and validation loss
    plt.subplot(1, 3, 1)
    if 'train_loss' in history:
        plt.plot(history['train_loss'], label='Training Loss', alpha=0.8)
    if 'val_loss' in history:
        plt.plot(history['val_loss'], label='Validation Loss', alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Validation accuracy
    plt.subplot(1, 3, 2)
    if 'val_accuracy' in history:
        plt.plot(history['val_accuracy'], label='Validation Accuracy', color='green', alpha=0.8)
        plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random Baseline')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Learning rate
    plt.subplot(1, 3, 3)
    if 'learning_rate' in history:
        plt.plot(history['learning_rate'], label='Learning Rate', color='orange', alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Schedule')
    plt.yscale('log')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    if 'train_loss' in history and history['train_loss']:
        print(f"Final training loss: {history['train_loss'][-1]:.4f}")
    if 'val_loss' in history and history['val_loss']:
        print(f"Final validation loss: {history['val_loss'][-1]:.4f}")
    if 'val_accuracy' in history and history['val_accuracy']:
        print(f"Final validation accuracy: {history['val_accuracy'][-1]:.4f}")
        
else:
    print("No training history found. Run training first.")
    
    # Show mock training curve
    epochs = np.arange(50)
    mock_loss = 2.0 * np.exp(-epochs/20) + 0.5 + 0.1 * np.random.randn(50)
    mock_acc = 0.5 + 0.3 * (1 - np.exp(-epochs/15)) + 0.05 * np.random.randn(50)
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, mock_loss, label='Training Loss', alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Expected Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, mock_acc, label='Validation Accuracy', color='green', alpha=0.8)
    plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random Baseline')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Expected Validation Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Mock training curves shown (run actual training for real results)")

## Evaluation Results Analysis

In [ ]:
# Load evaluation results if available
eval_path = Path('../evaluation_results/comprehensive_results.json')

if eval_path.exists():
    with open(eval_path, 'r') as f:
        eval_results = json.load(f)
    
    print("Evaluation results found!")
    
    # Extract metrics
    metrics = eval_results.get('metrics', {})
    target_comparisons = eval_results.get('target_comparisons', {})
    
    # Create metrics visualization
    if target_comparisons:
        metric_names = list(target_comparisons.keys())
        achieved_values = [comp['achieved'] for comp in target_comparisons.values()]
        target_values = [comp['target'] for comp in target_comparisons.values()]
        meets_target = [comp['meets_target'] for comp in target_comparisons.values()]
        
        # Metrics comparison chart
        plt.figure(figsize=(12, 6))
        
        x = np.arange(len(metric_names))
        width = 0.35
        
        bars1 = plt.bar(x - width/2, achieved_values, width, label='Achieved', alpha=0.8)
        bars2 = plt.bar(x + width/2, target_values, width, label='Target', alpha=0.8)
        
        # Color bars based on whether target is met
        for i, (bar, meets) in enumerate(zip(bars1, meets_target)):
            bar.set_color('green' if meets else 'red')
            bar.set_alpha(0.7)
        
        plt.xlabel('Metrics')
        plt.ylabel('Values')
        plt.title('Model Performance vs Targets')
        plt.xticks(x, [name.replace('_', '\n') for name in metric_names], rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        # Summary table
        print("\nMetrics Summary:")
        print("-" * 60)
        for name, comp in target_comparisons.items():
            status = "✓" if comp['meets_target'] else "✗"
            print(f"{name:30} {comp['achieved']:8.4f} / {comp['target']:8.4f} {status}")
        
        targets_met = sum(meets_target)
        total_targets = len(meets_target)
        print(f"\nTargets met: {targets_met}/{total_targets} ({100*targets_met/total_targets:.1f}%)")

else:
    print("No evaluation results found. Run evaluation first.")
    
    # Show target metrics
    target_metrics = config.get('evaluation.target_metrics', {})
    
    if target_metrics:
        print("\nTarget Metrics to Achieve:")
        print("-" * 40)
        for metric, target in target_metrics.items():
            print(f"{metric:30} {target:8.4f}")
            
        # Visualize targets
        plt.figure(figsize=(10, 6))
        metrics_list = list(target_metrics.keys())
        targets_list = list(target_metrics.values())
        
        plt.barh(metrics_list, targets_list, alpha=0.7)
        plt.xlabel('Target Values')
        plt.title('Target Metrics for Model Performance')
        plt.tight_layout()
        plt.show()

## Steering Module Visualization

In [ ]:
# Analyze steering module behavior
from preference_guided_diffusion_steering.models.model import SteeringModule

# Create steering module
steering_config = config.get('model.steering_config')
steering = SteeringModule(**steering_config)

# Analyze parameter distribution
all_params = []
layer_names = []
layer_sizes = []

for name, param in steering.named_parameters():
    if 'weight' in name:
        all_params.extend(param.flatten().detach().numpy())
        layer_names.append(name.replace('.weight', ''))
        layer_sizes.append(param.numel())

# Parameter distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.hist(all_params, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Parameter Value')
plt.ylabel('Frequency')
plt.title('Parameter Distribution')

plt.subplot(1, 3, 2)
plt.bar(range(len(layer_sizes)), layer_sizes, alpha=0.7)
plt.xlabel('Layer Index')
plt.ylabel('Number of Parameters')
plt.title('Parameters per Layer')
plt.xticks(range(len(layer_names)), [name.split('.')[-1] for name in layer_names], rotation=45)

plt.subplot(1, 3, 3)
# Show guidance scale behavior
guidance_scales = [0.01, 0.05, 0.1, 0.2, 0.5]
example_output_norms = []

# Mock analysis of output norms with different guidance scales
torch.manual_seed(42)
sample_input = torch.randn(1, 77, 768)

for scale in guidance_scales:
    with torch.no_grad():
        steering.guidance_scale.fill_(scale)
        output = steering(sample_input)
        norm = torch.norm(output).item()
        example_output_norms.append(norm)

plt.plot(guidance_scales, example_output_norms, 'o-', alpha=0.8)
plt.xlabel('Guidance Scale')
plt.ylabel('Output Norm')
plt.title('Guidance Scale Impact')
plt.xscale('log')

plt.tight_layout()
plt.show()

print(f"Total parameters: {sum(layer_sizes):,}")
print(f"Parameter std: {np.std(all_params):.6f}")
print(f"Parameter range: [{np.min(all_params):.6f}, {np.max(all_params):.6f}]")

## Preference Data Analysis

In [ ]:
# Analyze preference patterns
if len(dataset) > 0:
    prompts = [pair['prompt'] for pair in dataset.preference_pairs]
    rating_diffs = [pair.get('rating_diff', 1.0) for pair in dataset.preference_pairs]
    
    # Word frequency analysis
    from collections import Counter
    import re
    
    all_words = []
    for prompt in prompts:
        words = re.findall(r'\b\w+\b', prompt.lower())
        all_words.extend(words)
    
    word_counts = Counter(all_words)
    common_words = word_counts.most_common(20)
    
    plt.figure(figsize=(15, 5))
    
    # Word frequency
    plt.subplot(1, 3, 1)
    words, counts = zip(*common_words)
    plt.barh(range(len(words)), counts, alpha=0.7)
    plt.yticks(range(len(words)), words)
    plt.xlabel('Frequency')
    plt.title('Most Common Words in Prompts')
    plt.gca().invert_yaxis()
    
    # Rating difference vs prompt length
    plt.subplot(1, 3, 2)
    prompt_lens = [len(prompt.split()) for prompt in prompts]
    plt.scatter(prompt_lens, rating_diffs, alpha=0.6)
    plt.xlabel('Prompt Length (words)')
    plt.ylabel('Rating Difference')
    plt.title('Rating Difference vs Prompt Length')
    
    # Preference strength distribution
    plt.subplot(1, 3, 3)
    strength_bins = ['Weak (1.0-1.5)', 'Medium (1.5-2.5)', 'Strong (2.5+)']
    weak = sum(1 for rd in rating_diffs if 1.0 <= rd < 1.5)
    medium = sum(1 for rd in rating_diffs if 1.5 <= rd < 2.5)
    strong = sum(1 for rd in rating_diffs if rd >= 2.5)
    
    counts = [weak, medium, strong]
    colors = ['lightcoral', 'gold', 'lightgreen']
    
    plt.pie(counts, labels=strength_bins, colors=colors, autopct='%1.1f%%', startangle=90)
    plt.title('Preference Strength Distribution')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Vocabulary size: {len(word_counts)}")
    print(f"Average words per prompt: {np.mean(prompt_lens):.1f}")
    print(f"Preference strength - Weak: {weak}, Medium: {medium}, Strong: {strong}")

## Next Steps

In [ ]:
print("Next Steps for Preference-Guided Diffusion Steering:")
print("=" * 60)
print("")
print("1. Training:")
print("   Run: python scripts/train.py")
print("   - Monitors training progress with MLflow")
print("   - Saves checkpoints automatically")
print("   - Implements early stopping")
print("")
print("2. Evaluation:")
print("   Run: python scripts/evaluate.py --model-path checkpoints/best_checkpoint.pt")
print("   - Tests against target metrics")
print("   - Generates comparison images")
print("   - Measures latency overhead")
print("")
print("3. Target Metrics to Achieve:")
for metric, target in config.get('evaluation.target_metrics', {}).items():
    print(f"   - {metric}: {target}")
print("")
print("4. Hyperparameter Tuning:")
print("   - Adjust learning rate and batch size")
print("   - Experiment with steering module architecture")
print("   - Test different preference weighting strategies")
print("")
print("5. Advanced Analysis:")
print("   - Compare generated images qualitatively")
print("   - Analyze failure cases")
print("   - Conduct ablation studies")